In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
import os
import json

kaggle_creds = {
    "username": "YOUR_KAGGLE_USERNAME",
    "key": "YOUR_KAGGLE_KEY"
}

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)
os.system('chmod 600 /root/.kaggle/kaggle.json')

print("Done ✅")

In [ ]:
os.system('pip install kaggle -q')
os.system('kaggle datasets download -d pythonafroz/solar-panel-images-for-deep-learning --unzip -p /content/solar_panels')
print(os.listdir('/content/solar_panels'))

In [ ]:
import subprocess

result = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d',
     'pythonafroz/solar-panel-images-for-deep-learning',
     '--unzip', '-p', '/content/solar_panels'],
    capture_output=True, text=True
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)

In [ ]:
os.system('kaggle datasets download -d pythonafroz/solar-panel-images-for-deep-learning --unzip -p /content/solar_panels')
print(os.listdir('/content/solar_panels'))

In [ ]:
import os, json

kaggle_creds = {
    "username": "TON_USERNAME_KAGGLE",
    "key": "TON_TOKEN_ICI"
}

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)
os.system('chmod 600 /root/.kaggle/kaggle.json')
os.system('pip install kaggle -q')

os.system('kaggle datasets download -d alicjalenarczyk/pv-panel-defect-dataset --unzip -p /content/solar_panels')

print(os.listdir('/content/solar_panels'))

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile, os

with zipfile.ZipFile('archive.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/solar_panels')

for split in ['train', 'val', 'test']:
    classes = os.listdir(f'/content/solar_panels/{split}')
    print(f"{split} : {classes}")

In [ ]:
import os

print(os.listdir('/content/solar_panels'))

In [ ]:
print(os.listdir('/content/solar_panels/archive'))

In [ ]:
for split in ['train', 'val', 'test']:
    classes = os.listdir(f'/content/solar_panels/archive/{split}')
    print(f"{split} : {classes}")

In [ ]:
import torch
import torchvision
from torchvision import transforms, datasets, models
from torch import nn, optim
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

BASE = '/content/solar_panels/archive'

train_data = datasets.ImageFolder(BASE + '/train', transform=transform)
val_data   = datasets.ImageFolder(BASE + '/val',   transform=transform)
test_data  = datasets.ImageFolder(BASE + '/test',  transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=32, shuffle=False)

print("Classes:", train_data.classes)
print("Train:", len(train_data), "| Val:", len(val_data), "| Test:", len(test_data))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using:", device)

model = models.resnet50(weights='IMAGENET1K_V1')

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 6)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

print("Model ready")

In [ ]:
for epoch in range(10):
    model.train()
    total_loss, correct = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    train_acc = correct / len(train_data) * 100

    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()

    val_acc = correct / len(val_data) * 100
    print(f"Epoch {epoch+1}/10 | Loss: {total_loss/len(train_loader):.3f} | Train: {train_acc:.1f}% | Val: {val_acc:.1f}%")

In [ ]:
for param in model.layer4.parameters():
    param.requires_grad = True

optimizer = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 0.0001},
    {'params': model.fc.parameters(),     'lr': 0.001}
])

for epoch in range(10):
    model.train()
    total_loss, correct = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    train_acc = correct / len(train_data) * 100

    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()

    val_acc = correct / len(val_data) * 100
    print(f"Epoch {epoch+1}/10 | Loss: {total_loss/len(train_loader):.3f} | Train: {train_acc:.1f}% | Val: {val_acc:.1f}%")

In [ ]:
torch.save(model.state_dict(), 'solar_model.pth')
print("Model saved")

In [ ]:
model.eval()
correct = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        correct += (outputs.argmax(1) == labels).sum().item()

print(f"Test accuracy: {correct / len(test_data) * 100:.1f}%")

In [ ]:
!pip install groq -q

import os
from groq import Groq

os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

client = Groq()

def generate_report(defect_class, confidence):
    prompt = f"""You are a solar panel maintenance expert.
A computer vision model detected: {defect_class} (confidence: {confidence:.1f}%).

Write a short maintenance report with:
- Defect severity (Low / Medium / High)
- Recommended action
- Estimated production loss

Be concise, max 5 sentences."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

report = generate_report("Physical-Damage", 94.7)
print(report)

In [ ]:
app_code = '''
import streamlit as st
import torch
from torchvision import transforms, models
from torch import nn
from PIL import Image
from groq import Groq
import os

# Load model
@st.cache_resource
def load_model():
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 6)
    model.load_state_dict(torch.load("solar_model.pth", map_location="cpu"))
    model.eval()
    return model

CLASSES = ["Bird-drop", "Clean", "Dusty", "Electrical-damage", "Physical-Damage", "Snow-Covered"]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def predict(image, model):
    tensor = transform(image).unsqueeze(0)
    with torch.no_grad():
        output = model(tensor)
        probs = torch.softmax(output, dim=1)
        confidence, predicted = probs.max(1)
    return CLASSES[predicted.item()], confidence.item() * 100

def generate_report(defect_class, confidence):
    client = Groq(api_key=os.environ["GROQ_API_KEY"])
    prompt = f"""You are a solar panel maintenance expert.
A computer vision model detected: {defect_class} (confidence: {confidence:.1f}%).
Write a short maintenance report with:
- Defect severity (Low / Medium / High)
- Recommended action
- Estimated production loss
Be concise, max 5 sentences."""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# UI
st.title("Solar Defect Inspector")
st.write("Upload a solar panel image to detect defects and generate a maintenance report.")

uploaded = st.file_uploader("Upload an image", type=["jpg", "jpeg", "png"])

if uploaded:
    image = Image.open(uploaded).convert("RGB")
    st.image(image, caption="Uploaded image", use_column_width=True)

    model = load_model()
    defect, confidence = predict(image, model)

    st.subheader("Detection Result")
    if defect == "Clean":
        st.success(f"No defect detected ({confidence:.1f}% confidence)")
    else:
        st.error(f"Defect detected: **{defect}** ({confidence:.1f}% confidence)")
        st.subheader("Maintenance Report")
        with st.spinner("Generating report..."):
            report = generate_report(defect, confidence)
        st.write(report)
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py created")

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

!pip install streamlit groq -q
!npm install -g localtunnel -q

!pkill -f streamlit

import subprocess, threading, time

def run_streamlit():
    subprocess.run(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

time.sleep(5)

import subprocess
result = subprocess.Popen(
    ["lt", "--port", "8501"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)
time.sleep(3)
print(result.stdout.readline())

In [ ]:
import subprocess
result = subprocess.run(["python", "-c", "import app"],
                       capture_output=True, text=True, cwd="/content")
print(result.stderr)

In [ ]:
import os, subprocess, threading, time

os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

!pkill -f streamlit
!pkill -f lt
time.sleep(2)

def run_streamlit():
    subprocess.run(["streamlit", "run", "app.py",
                   "--server.port", "8501",
                   "--server.headless", "true",
                   "--server.enableCORS", "false",
                   "--server.enableXsrfProtection", "false"])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()
time.sleep(8)

tunnel = subprocess.Popen(["lt", "--port", "8501"],
                          stdout=subprocess.PIPE, text=True)
time.sleep(3)
url = tunnel.stdout.readline().strip()
print("URL:", url)

In [ ]:
import os, subprocess, threading, time

os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

!pkill -f streamlit
!pkill -f cloudflared
time.sleep(2)

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

def run_streamlit():
    subprocess.run(["streamlit", "run", "app.py",
                   "--server.port", "8501",
                   "--server.headless", "true",
                   "--server.enableCORS", "false",
                   "--server.enableXsrfProtection", "false"])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()
time.sleep(8)

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

for line in tunnel.stderr:
    if "trycloudflare.com" in line:
        url = line.split("https://")[-1].strip()
        print("URL: https://" + url)
        break

In [ ]:
for line in tunnel.stderr:
    if "trycloudflare.com" in line and "INF" in line and "http" in line:
        import re
        match = re.search(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', line)
        if match:
            print("URL:", match.group())
            break

In [ ]:
from google.colab import files
files.download('app.py')
files.download('solar_model.pth')